# Coffee17 preprocessing — final analysis
Paired bootstrap, per-class/confusion, efficiency, final report. No training.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path
BRANCH='codex/preprocessing-study-v1'; REPO=Path('/content/coffee-bean-classification'); WORK=Path('/content')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-classification.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements/preprocessing-study.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from bilinear_lmmd.core.drive_project import resolve_drive_project_root
from bilinear_lmmd.data.preparation.prepare_coffee17 import DATASET_URL
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
PROJECT=resolve_drive_project_root(); OOF=PROJECT/'oof/coffee17-preprocessing-primary-v1/merged'; ANALYSIS=PROJECT/'analysis/coffee17-preprocessing-primary-v1'; ANALYSIS.mkdir(parents=True,exist_ok=True)
MASTER=OOF/'primary_oof_table.csv'; SUMMARY=OOF/'primary_oof_summary.json'
if not MASTER.is_file() or not SUMMARY.is_file(): raise FileNotFoundError('OOF merged evidence belum lengkap')
subprocess.run([sys.executable,'-u','-m','bilinear_lmmd.experiments.run_preprocessing_bootstrap','--master-table',str(MASTER),'--output',str(ANALYSIS/'paired_bootstrap.json'),'--iterations','10000'],check=True)
subprocess.run([sys.executable,'-u','-m','bilinear_lmmd.experiments.run_preprocessing_analysis','--master-table',str(MASTER),'--output-dir',str(ANALYSIS)],check=True)
ARCHIVE=WORK/'coffee17_original.zip'
if not ARCHIVE.is_file():
    req=urllib.request.Request(DATASET_URL,headers={'User-Agent':'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response, ARCHIVE.open('wb') as output: shutil.copyfileobj(response,output)
CANONICAL=WORK/'coffee17_original_v1'; PROV=WORK/'coffee17_analysis_provenance'
if CANONICAL.exists(): shutil.rmtree(CANONICAL)
if PROV.exists(): shutil.rmtree(PROV)
audit_coffee17_provenance(ARCHIVE,PROV,canonical_root=CANONICAL)
subprocess.run([sys.executable,'-u','-m','bilinear_lmmd.experiments.run_preprocessing_efficiency','--canonical-root',str(CANONICAL),'--clean-manifest',str(PROJECT/'evidence/coffee17-preprocessing-data-v1/clean_manifest.json'),'--output',str(ANALYSIS/'preprocessing_efficiency.json'),'--batch-sizes','1','16','--warmup','10','--iterations','50'],check=True)
subprocess.run([sys.executable,'-u','-m','bilinear_lmmd.experiments.run_preprocessing_final_report','--oof-summary',str(SUMMARY),'--bootstrap',str(ANALYSIS/'paired_bootstrap.json'),'--analysis-summary',str(ANALYSIS/'analysis_summary.json'),'--efficiency',str(ANALYSIS/'preprocessing_efficiency.json'),'--output-dir',str(ANALYSIS)],check=True)
print('FINAL:',ANALYSIS/'FINAL_PREPROCESSING_REPORT.md')
